# Estimation of the three feedstocks

Solves the nonlinear program of Section 3.4 (Algorithm 1) for rice husk, the 50:50 blend and corncob. The search of the article (two sequential passes and 40 independent admissible starts per feedstock) takes about half an hour; its best decision vectors ship with the package, and `random_starts=0` reproduces the reported estimate from them in a few minutes. The complete set of result tables is written by `python -m ccpyro.model`.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "02_estimation_and_figures"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import model as mp
import matplotlib.pyplot as plt
built = mp.build_cases(random_starts=0)
states, contexts = built["states"], built["contexts"]
for label, s in states.items():
    print(f"{label:24s} objective {s.objective:.5f}, smallest constraint {mp.constraints(s.vector, contexts[label]).min():.1e}")

## Conservation

The residuals of the carbon, hydrogen and oxygen balances vanish at machine precision, because the balances are eliminated in closed form (Eqs. 8-10).

In [ ]:
pd.DataFrame({label: mp.verification(s, contexts[label]) for label, s in states.items()}).T

## Products and tar composition

In [ ]:
rows = []
for label, s in states.items():
    for k, T in enumerate(mp.T_NODES):
        rows.append({"treatment": label, "T": T, "w_char": s.w_char[k], "w_tar": s.w_tar[k],
                     "w_water": s.w_water[k], "w_gas": s.w_gas[k], "x_t": s.x_t[k], "y_t": s.y_t[k],
                     "x_c": s.x_c[k], "y_c": s.y_c[k]})
table = pd.DataFrame(rows)
table.round(4)

In [ ]:
published = ARTICLE / "product_yields.csv"
if published.exists():
    ref = pd.read_csv(published)
    merged = table.merge(ref, on=["treatment", "T"], suffixes=("", "_article"))
    cols = ["w_char", "w_tar", "w_water", "w_gas", "x_t", "y_t", "x_c", "y_c"]
    diff = max(float((merged[c] - merged[c + "_article"]).abs().max()) for c in cols)
    print(f"largest difference from the article: {diff:.1e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (label, s) in zip(axes, states.items()):
    for key, name in (("w_char", "char"), ("w_tar", "tar"), ("w_water", "water"), ("w_gas", "gas")):
        ax.plot(mp.T_NODES, getattr(s, key), marker="o", label=name)
    ax.plot(mp.T_NODES, contexts[label].theta, "k--", lw=1, label="residual mass (TGA)")
    ax.set_title(label); ax.set_xlabel("T (°C)")
axes[0].set_ylabel("mass fraction, daf"); axes[0].legend(fontsize=8)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for label, s in states.items():
    line, = ax.plot(s.y_t, s.x_t, marker="o", label=f"tar, {label.split(' (')[0]}")
    ax.plot(s.y_c, s.x_c, marker="s", ls="--", color=line.get_color(), alpha=0.6)
    ax.plot(contexts[label].y0, contexts[label].x0, marker="*", ms=12, color=line.get_color())
ax.set_xlabel("O/C (atomic)"); ax.set_ylabel("H/C (atomic)"); ax.legend(fontsize=8)
ax.set_title("Van Krevelen diagram: tar (solid), char (dashed), feedstock (star)");

## Measured and modelled gas

In [ ]:
label = "rice husk (pelletized)"
s, ctx = states[label], contexts[label]
pd.DataFrame(np.hstack([s.phi_model, ctx.composition]), index=mp.T_NODES,
             columns=[f"{g} model" for g in mp.GAS_SPECIES] + [f"{g} measured" for g in mp.GAS_SPECIES]).round(4)